# SN-01 — Sirenisation Phase 1 (SIREN exact)

Pour chaque EJ FINESS ayant un `nmsiren_stru`, lookup direct dans la base UL SIRENE complète. Cette phase produit notre **liste des SIREN validés** (VALIDE_FORT + VALIDE), qui sera utilisée pour exclure ces EJ du périmètre A/B/C.

Statuts : VALIDE_FORT / VALIDE / DOUTEUX / REJETE / SANS_SIREN / SIREN_INCONNU

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.sirenisation import matching_direct_siren
from src.excel_export import export_phase1_excel, LABELS
from src.display      import afficher_tableau, afficher_synthese
from config.settings  import (
    FINESS_EJ_CLEAN, SIRENE_UL_CLEAN, SN_PHASE1, RESULTS_SN_DIR,
)

RESULTS_SN_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement

In [2]:
df_ej = pd.read_parquet(FINESS_EJ_CLEAN)
df_ul = pd.read_parquet(SIRENE_UL_CLEAN)

df_ej['nmsiren_stru'] = df_ej['nmsiren_stru'].fillna('').astype(str)
df_ul['siren']        = df_ul['siren'].astype(str)

print(f'EJ FINESS : {len(df_ej):,}')
print(f'UL SIRENE : {len(df_ul):,}')

EJ FINESS : 54,185
UL SIRENE : 15,111,205


## 2. Matching SIREN exact

In [5]:
df_resultats = matching_direct_siren(df_ej, df_ul, desc='Matching SIREN exact')
print(df_resultats['statut'].value_counts())

Matching SIREN exact:   0%|          | 0/54185 [00:00<?, ?it/s]

statut
VALIDE_FORT      23059
VALIDE           15551
DOUTEUX           6712
SIREN_INCONNU     4103
REJETE            2660
SANS_SIREN        2100
Name: count, dtype: int64


## 3. Enrichissement avec colonnes UL SIRENE

In [6]:
COLS_UL_JOIN = ['siren', 'denominationUniteLegale', 'sigleUniteLegale',
                'categorieJuridiqueUniteLegale', 'activitePrincipaleUniteLegale',
                'dateCreationUniteLegale',
                'adresse_siege_complete_ul', 'codeCommuneEtablissement']
ul_join = df_ul[[c for c in COLS_UL_JOIN if c in df_ul.columns]].drop_duplicates('siren').copy()
ul_join['siren'] = ul_join['siren'].astype(str)

df_resultats['siren_ul'] = df_resultats['siren_ul'].astype(str)
df_resultats = df_resultats.merge(
    ul_join, left_on='siren_ul', right_on='siren', how='left',
).drop(columns=['siren'], errors='ignore')

## 4. Aperçu

In [7]:
afficher_tableau(
    df_resultats[df_resultats['statut'].isin(['VALIDE_FORT', 'VALIDE'])],
    'Aperçu validés Phase 1', max_lignes=9,
    colonnes=['idstructure_stru', 'nmsiren_stru', 'raisonsociale_stru',
              'denominationUniteLegale', 'nom_ul_retenu',
              'score_nom', 'score_adresse', 'score_global', 'statut'],
)

idstructure_stru,nmsiren_stru,raisonsociale_stru,denominationUniteLegale,nom_ul_retenu,score_nom,score_adresse,score_global,statut
1831435,333483667,HABITAT PLURIEL,HABITAT PLURIEL,HABITAT PLURIEL,100.000000,100.000000,100.000000,VALIDE_FORT
1831436,782974158,ASSOCIATION LE CANA,CENTRE FORMATION PREPARATION A L'EMPLOI,CENTRE FORMATION PREPARATION EMPLOI,23.380000,100.000000,69.350000,VALIDE
1831438,334353471,ASSOCIATION REGIONALE POUR INTEGRATION,ASSOCIATION REGIONALE POUR L INTEGRATION,REGIONALE INTEGRATION,100.000000,100.000000,100.000000,VALIDE_FORT
1831440,775559701,ENTRAIDE,ENTRAIDE,ENTRAIDE,100.000000,100.000000,100.000000,VALIDE_FORT
1831441,775558364,CAF 13,CAISSE D'ALLOCATIONS FAMILIALES DES BOUCHES DU RHONE,CAF 13,100.000000,100.000000,100.000000,VALIDE_FORT
1831443,782885735,CPAM 13,CAISSE PRIMAIRE CENTRALE ASSUR MALADIE,CAISSE PRIMAIRE CENTRALE ASSUR MALADIE,17.830000,100.000000,67.130000,VALIDE
1831444,775560105,ASSOCIATION MEDICO-SOCIALE DE PROVENCE,MEDICO SOCIALE DE PROVENCE,MEDICO SOCIALE PROVENCE,100.000000,100.000000,100.000000,VALIDE_FORT
1831445,775559719,SAUVEGARDE 13,SAUVEGARDE 13,SAUVEGARDE 13,100.000000,100.000000,100.000000,VALIDE_FORT
1831447,775558968,ASSOCIATION UNAPEI ALPES PROVENCE,UNAPEI ALPES PROVENCE,UNAPEI ALPES PROVENCE,100.000000,100.000000,100.000000,VALIDE_FORT


## 5. Export Excel

In [8]:
COLS_COMPLET = [
    'idstructure_stru', 'nmfinessej_stru', 'nmfinessetab_stru', 
    'categetab_stru', 'nmsiren_stru', 'raisonsociale_stru',
    'cdape_stru', 'dtouvertstruct_stru',
    'cdcommune_stru', 'adresse_complete_ej',
    'siren_ul', 'denominationUniteLegale', 'sigleUniteLegale',
    'nom_ul_retenu', 'adresse_siege_complete_ul', 'codeCommuneEtablissement',
    'categorieJuridiqueUniteLegale', 'activitePrincipaleUniteLegale',
    'dateCreationUniteLegale',
    'score_nom', 'score_adresse', 'score_global',
]
COLS_INFO = [
    'idstructure_stru', 'nmsiren_stru', 'raisonsociale_stru',
    'cdape_stru', 'dtouverture_stru',
    'cdcommune_stru', 'adresse_complete_ej',
]

compteurs = export_phase1_excel(
    df_resultats, SN_PHASE1, COLS_COMPLET, COLS_INFO,
    statuts_score=['VALIDE_FORT', 'VALIDE', 'DOUTEUX', 'REJETE'],
    statuts_info =['SANS_SIREN', 'SIREN_INCONNU'],
)

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  'Synthèse Sirenisation Phase 1')
print(f'\nFichier : {SN_PHASE1}')

Statut,Nb,% du total
Valide_fort,"23,059",42.6%
Valide,"15,551",28.7%
Douteux,"6,712",12.4%
Rejeté,"2,660",4.9%
Sans_SIREN,"2,100",3.9%
SIREN_inconnu,"4,103",7.6%
TOTAL,"54,185",100.0%



Fichier : /home/jovyan/work/projet_finess_sirene/results/sirenisation/sirenisation_phase1.xlsx
